<a href="https://colab.research.google.com/github/bcalm-2/gen-ai-assignments-3.4.5/blob/main/Srashti_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate pillow -q

import torch, os, time
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

os.makedirs("generated_images", exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "nota-ai/bk-sdm-tiny",
    torch_dtype=torch.float32,
    safety_checker=None,
).to("cpu")

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing(1)

prompts = [
    "A realistic sunset over mountains with cinematic lighting",
    "A cute cartoon cat playing with a ball",
    "A modern poster design with bold typography saying AI Future",
    "A red sports car on a highway at night",
    "A futuristic city with flying cars and neon lights"
]

all_images = []   # collect as we go
all_titles = []

with torch.inference_mode():
    for i, prompt in enumerate(prompts):
        t0 = time.time()
        pair = []
        for j in range(2):
            image = pipe(
                prompt,
                num_inference_steps=15,
                guidance_scale=6.0,
                height=256,
                width=256,
            ).images[0]
            fname = f"generated_images/SD_prompt{i+1}_img{j+1}.png"
            image.save(fname)
            pair.append(image)
            all_images.append(image)
            all_titles.append(f"P{i+1}-img{j+1}")

        elapsed = time.time() - t0
        print(f"Prompt {i+1} done | {elapsed:.1f}s")

        # Show all completed images so far after each prompt
        clear_output(wait=True)
        completed = len(all_images)
        cols = 2
        rows = (completed + 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(8, rows * 4))
        axes = axes.flatten() if completed > 2 else [axes] if completed == 1 else axes.flatten()

        for k, (img, title) in enumerate(zip(all_images, all_titles)):
            axes[k].imshow(img)
            axes[k].set_title(title, fontsize=9)
            axes[k].axis("off")

        for k in range(len(all_images), len(axes)):
            axes[k].axis("off")

        plt.suptitle(f"Generated so far: {completed}/10", fontsize=12)
        plt.tight_layout()
        plt.show()

# Final save
fig, axes = plt.subplots(5, 2, figsize=(10, 25))
for idx, (img, title) in enumerate(zip(all_images, all_titles)):
    axes[idx // 2][idx % 2].imshow(img)
    axes[idx // 2][idx % 2].set_title(title, fontsize=9)
    axes[idx // 2][idx % 2].axis("off")
plt.suptitle("All Generated Images", fontsize=14)
plt.tight_layout()
plt.savefig("generated_images/grid_output.png", bbox_inches="tight", dpi=150)
plt.show()
print("Final grid saved.")

In [ ]:
!pip install openai pillow -q

import os, base64
from openai import OpenAI
from PIL import Image
from io import BytesIO
from google.colab import userdata
import matplotlib.pyplot as plt

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

os.makedirs("generated_images", exist_ok=True)

prompts = [
    "A realistic sunset over mountains with cinematic lighting",
    "A cute cartoon cat playing with a ball",
    "A modern poster design with bold typography saying AI Future",
    "A red sports car on a highway at night",
    "A futuristic city with flying cars and neon lights"
]

images, titles = [], []

for i, prompt in enumerate(prompts):
    try:
        print(f"Generating {i+1}/5...")
        result = client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            size="1024x1024",
            quality="standard",
            response_format="b64_json",
            n=1,
        )
        img = Image.open(BytesIO(base64.b64decode(result.data[0].b64_json)))
        img.save(f"generated_images/DALLE_prompt{i+1}.png")
        images.append(img)
        titles.append(f"P{i+1}: {prompt[:35]}...")
        print(f"  Done.")
    except Exception as e:
        print(f"  Failed: {e}")

fig, axes = plt.subplots(3, 2, figsize=(12, 18))
axes = axes.flatten()
for k in range(6):
    if k < len(images):
        axes[k].imshow(images[k])
        axes[k].set_title(titles[k], fontsize=9, wrap=True)
    axes[k].axis("off")
plt.suptitle("DALL-E 3 — Generated Images", fontsize=13)
plt.tight_layout()
plt.savefig("generated_images/DALLE_grid.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Done. {len(images)}/5 images generated.")

In [ ]:
# ACTIVITY 2: HUMAN EVALUATION

import pandas as pd

# Create evaluation table
data = {
    "Image": [
        "SD_prompt1_img1",
        "SD_prompt1_img2",
        "SD_prompt2_img1"
    ],
    "Prompt Adherence (1-5)": [4, 5, 4],
    "Visual Quality (1-5)": [4, 5, 3],
    "Realism (1-5)": [5, 4, 3],
    "Consistency (1-5)": [4, 5, 4]
}

df = pd.DataFrame(data)
print(df)

# Save table
df.to_csv("evaluation_table.csv", index=False)

In [ ]:
# ACTIVITY 3: MODEL COMPARISON

import pandas as pd

comparison = {
    "Criteria": [
        "Prompt Adherence",
        "Visual Quality",
        "Style Control",
        "Artifacts"
    ],
    "Stable Diffusion": [
        "Good",
        "High",
        "Flexible",
        "Some artifacts"
    ],
    "DALL·E": [
        "Very Good",
        "Very High",
        "Better control",
        "Less artifacts"
    ]
}

df = pd.DataFrame(comparison)
print(df)

df.to_csv("comparison_table.csv", index=False)